# Spot–perp cash-and-carry — crypto v2 (market-neutral funding harvest)

**Slug:** `spot-perp-carry`  ·  **Date:** 2026-06-03  ·  **Status:** `accept_monitoring`
**Decision record:** `docs/research_decisions/crypto_intraday/P6-spot-perp-carry.md`

First *market-neutral* strategy in the crypto engagement. Built after v1 (P0–P5) showed every
directional/5m edge was either cost-killed or a bull-beta proxy that died in the bear.
Requires the editable install (`pip install -e .`) so `import alpha_lab` resolves.

## Research question
Does a delta-neutral **long-spot / short-perp** book on BTC & ETH that harvests positive perp funding
(rebalanced daily) earn a **positive net-of-cost return in every regime — bear, bull, and the 2025
out-of-sample window** — at conservative perp+spot "stress" costs?

## Hypothesis & economic rationale
**Carry / liquidity provision.** Positive perp funding is paid by longs to shorts. Holding
**short perp + long spot** of the same asset is delta-neutral (price legs cancel), so the book
**collects funding with ~zero directional exposure** → regime-independent by construction.
**Skeptic:** funding has compressed; absolute yield is modest; a naive bar-by-bar implementation
churns 4 legs and pays the carry back in fees. The edge only exists if the position is *held*.

In [ ]:
import numpy as np
import pandas as pd
from alpha_lab.data.loaders.binance_vision import load_klines, load_funding
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.backtest.metrics import summary

SYMS = ["BTCUSDT", "ETHUSDT"]
START, END = "2021-10-01", "2026-01-01"   # END exclusive -> 2026 PM holdout never loaded
BPY = 24 * 365
YEARS = [2022, 2023, 2024, 2025]          # 2025 = research-test OOS
# per-leg total bps (fee + slippage); folded into slippage so costs_bps=0
STRESS = {"BTC.p": 8.0, "ETH.p": 10.0, "BTC.s": 15.0, "ETH.s": 17.5}
BASE   = {"BTC.p": 6.0, "ETH.p": 6.5,  "BTC.s": 12.0, "ETH.s": 13.0}

## Universe & data
BTC/ETH **perp + spot** 1h klines + perp funding (8h), 2021-10 → 2025-12, Binance Vision.
Note: the loader was patched to auto-detect ms/µs timestamps — Binance moved **spot** klines to
microseconds in 2025, which had silently dropped the entire OOS year.

In [ ]:
perp = load_klines(SYMS, "1h", START, END, market="perp").close_panel()
spot = load_klines(SYMS, "1h", START, END, market="spot").close_panel()
fund = load_funding(SYMS, START, END)
idx = perp.index.intersection(spot.index)
prices = pd.DataFrame({
    "BTC.p": perp["BTCUSDT"].reindex(idx), "ETH.p": perp["ETHUSDT"].reindex(idx),
    "BTC.s": spot["BTCUSDT"].reindex(idx), "ETH.s": spot["ETHUSDT"].reindex(idx),
}).dropna()
fund_p = fund.rename(columns={"BTCUSDT": "BTC.p", "ETHUSDT": "ETH.p"})
prices.shape, prices.index.min(), prices.index.max()

## Signal & portfolio (leak-safe, delta-neutral, held)
`active = (7-day trailing mean of funding, ffilled to the 1h grid using only events ≤ t) > thr`.
When active: `+0.25 spot, −0.25 perp` per symbol (gross ≤ 1); flat otherwise. No `shift(-k)`;
`run_backtest` lags weights +1 bar. **Daily** rebalance — the smoothed regime flips rarely, so the
position is *held* (turnover ≈ 18 over four years), which is what makes the carry survive cost.

In [ ]:
def carry_weights(thr=0.0, smooth_h=168, a=0.25, syms=("BTC", "ETH"), extra_lag_h=0):
    f = fund_p.reindex(prices.index, method="ffill")          # most recent funding <= t (leak-safe)
    if extra_lag_h:
        f = f.shift(extra_lag_h)                              # probe: extra delay should barely matter
    active = (f.rolling(smooth_h).mean() > thr).astype(float)  # trailing funding regime
    w = pd.DataFrame(0.0, index=prices.index, columns=prices.columns)
    for s in syms:
        w[f"{s}.s"] =  a * active[f"{s}.p"]                    # long spot
        w[f"{s}.p"] = -a * active[f"{s}.p"]                    # short perp -> collects funding
    return w

def bt(w, costs=STRESS):
    return run_backtest(w, prices, rebalance="D", costs_bps=0.0,
                        slippage_bps=costs, funding=fund_p, bars_per_year=BPY)

## Headline performance & per-year (across real regimes), net of stress costs

In [ ]:
res = bt(carry_weights())

def per_year(res):
    rows = []
    for y in YEARS:
        r = res.returns[res.returns.index.year == y]
        s = summary(r, periods=BPY)
        rows.append({"year": y, "Sharpe": round(s["Sharpe"], 2),
                     "net": float((1 + r).prod() - 1), "MaxDD": s["MaxDD"]})
    return pd.DataFrame(rows).set_index("year")

print({k: round(v, 3) for k, v in summary(res.returns, periods=BPY).items()})
per_year(res)

## Diagnostics — attribution proves market-neutrality
The price legs should contribute ≈ 0: if so, the PnL *is* the funding carry minus cost, not a hidden
directional bet (the v1 failure mode).

In [ ]:
print(f"funding received {-res.funding_costs.sum():+.1%} | "
      f"cost {res.costs.sum():+.1%} | "
      f"price-leg delta {res.gross_returns.sum():+.2%} | "
      f"net {float((1 + res.returns).prod() - 1):+.1%} | "
      f"turnover {res.turnover.sum():.0f} | in-market {(res.weights.abs().sum(1) > 0).mean():.0%}")

## Robustness + leak probe (perturbations of one mechanism, not independent bets)

In [ ]:
def net(r):
    return float((1 + r.returns).prod() - 1)

print("BTC-only            ", f"{net(bt(carry_weights(syms=('BTC',)))):+.1%}")
print("ETH-only            ", f"{net(bt(carry_weights(syms=('ETH',)))):+.1%}")
for thr in (0.0, 1e-5, 3e-5, 5e-5):
    print(f"thr={thr:.0e}          ", f"{net(bt(carry_weights(thr=thr))):+.1%}")
for sm in (72, 168, 336):
    print(f"smooth={sm:>3d}h          ", f"{net(bt(carry_weights(smooth_h=sm))):+.1%}")
print("BASE costs          ", f"{net(bt(carry_weights(), costs=BASE)):+.1%}")
print("+24h lag (leak probe)", f"{net(bt(carry_weights(extra_lag_h=24))):+.1%}")

## Decision — `accept_monitoring`
Positive in the 2022 bear, the bull, **and** the 2025 OOS; market-neutral (price-leg ≈ 0); leak-free
(+24h lag ≈ no change); cost-insensitive (low turnover). *Not* full `accept`: absolute return is
modest (~3.7%/yr) and operational realism (perp margin/liquidation, stressed-unwind fills) is
unmodeled. **Monitor** realized funding yield + live execution cost; **kill** if trailing-quarter net
carry < 0 or a margin/liquidation event hits the perp leg. Full record + next steps in
`docs/research_decisions/crypto_intraday/P6-spot-perp-carry.md`. **PM holdout (2026) was not accessed.**

**Next:** lift the carry builder to `src/alpha_lab/backtest/crypto_carry.py` on second reuse; extend to
SOL/BNB perps; add a stressed-unwind execution model; pre-registered 2026 PM-holdout gate.